In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# MY CODE


In [2]:
import os
os.environ["WANDB_PROJECT"] = "23f3000843-t22026"
os.environ["WANDB_ENTITY"] = "varnitchourasiya27-indian-institute-of-technology-madras"

In [3]:
import os
import warnings
from dataclasses import dataclass
from typing import Optional, Union

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy

import wandb
from kaggle_secrets import UserSecretsClient

warnings.filterwarnings("ignore")

OPTIONS = ["A", "B", "C", "D", "E"]
LABEL_TO_IDX = {label: i for i, label in enumerate(OPTIONS)}
IDX_TO_LABEL = {i: label for i, label in enumerate(OPTIONS)}

In [4]:
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [5]:
secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: varnitchourasiya27 (varnitchourasiya27-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)

train_df_split, eval_df_split = train_test_split(train, test_size=0.1, random_state=42)
print("Train split:", train_df_split.shape, "| Eval split:", eval_df_split.shape)

Train shape: (2000, 8)
Test shape : (500, 7)
Train split: (1800, 8) | Eval split: (200, 8)


In [7]:
model_name = "google/electra-base-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMultipleChoice.from_pretrained(model_name)


def preprocess_multiple_choice(examples):
    first_sentences = [[prompt] * 5 for prompt in examples["prompt"]]
    second_sentences = [
        [examples["A"][i], examples["B"][i], examples["C"][i], examples["D"][i], examples["E"][i]]
        for i in range(len(examples["prompt"]))
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=256,
    )

    features = {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}
    features["labels"] = [LABEL_TO_IDX[a] for a in examples["answer"]]
    return features


train_dataset = Dataset.from_pandas(train_df_split.reset_index(drop=True)).map(
    preprocess_multiple_choice, batched=True
)
eval_dataset = Dataset.from_pandas(eval_df_split.reset_index(drop=True)).map(
    preprocess_multiple_choice, batched=True
)

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

ElectraForMultipleChoice LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings_project.bias                   | UNEXPECTED | 
classifier.weight                                 | MISSING    | 
classifier.bias                                   | MISSING    | 
sequence_summary.summary.weight                   | MISSING    | 
sequence_summary.summary.bias                     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [8]:
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch


In [9]:
import gc
torch.cuda.empty_cache()
gc.collect()

training_args = TrainingArguments(
    output_dir="./electra_mcq_results",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="wandb",
    run_name="electra-base-mcq-v2",
    logging_steps=10,
    seed=42,
    data_seed=42,
)

def compute_metrics(eval_predictions):
    from sklearn.metrics import f1_score
    predictions, label_ids = eval_predictions
    preds = np.argmax(predictions, axis=1)
    acc = (preds == label_ids).astype(np.float32).mean().item()
    f1  = f1_score(label_ids, preds, average="macro")
    return {"accuracy": acc, "f1": f1}

model.gradient_checkpointing_enable()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260723_094549-zk5oylzr
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run electra-base-mcq-v2
wandb: ⭐️ View project at https://wandb.ai/varnitchourasiya27-indian-institute-of-technology-madras/23f3000843-t22026
wandb: 🚀 View run at https://wandb.ai/varnitchourasiya27-indian-institute-of-technology-madras/23f3000843-t22026/runs/zk5oylzr


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.429347,0.244967,0.995000,0.993262
2,0.058981,0.009294,1.000000,1.000000
3,0.000495,0.000111,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=675, training_loss=0.6168884354715097, metrics={'train_runtime': 444.1442, 'train_samples_per_second': 12.158, 'train_steps_per_second': 1.52, 'total_flos': 1457108275009680.0, 'train_loss': 0.6168884354715097, 'epoch': 3.0})

In [10]:
def map_at_3(df, predict_fn):
    scores = []
    for _, row in df.iterrows():
        prediction = predict_fn(row)
        predicted_labels = prediction.split()
        correct = row["answer"]
        score = 0.0
        if correct in predicted_labels:
            rank = predicted_labels.index(correct) + 1
            score = 1.0 / rank
        scores.append(score)
    return np.mean(scores)


def evaluate_and_log(model_name, predict_fn, eval_df, sample_size=200):
    run = wandb.init(
        entity="varnitchourasiya27-indian-institute-of-technology-madras",
        project="23f3000843-t22026",
        name=model_name,
        config={"model": model_name, "sample_size": sample_size},
    )
    sample = eval_df.sample(min(sample_size, len(eval_df)), random_state=42)
    score = map_at_3(sample, predict_fn)
    wandb.log({"MAP@3": score})
    print(f"{model_name} -> Local MAP@3 (held-out): {score:.4f}")
    wandb.finish()
    return score


def electra_predict_fn(row):
    first_sentences = [row["prompt"]] * 5
    second_sentences = [row["A"], row["B"], row["C"], row["D"], row["E"]]

    inputs = tokenizer(
        first_sentences, second_sentences,
        truncation=True, max_length=256, padding=True,
        return_tensors="pt",
    )
    inputs = {k: v.unsqueeze(0).to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits.squeeze(0)
    probs = F.softmax(logits, dim=0)

    top3_idx = torch.argsort(probs, descending=True)[:3]
    return " ".join([IDX_TO_LABEL[i] for i in top3_idx.tolist()])


score = evaluate_and_log("electra-base-mcq-v2", electra_predict_fn, eval_df=eval_df_split, sample_size=200)


wandb: Finishing previous runs because reinit is set to 'default'.
wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 68-70, summary, console lines 0-0
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁██
wandb:                 eval/f1 ▁██
wandb:               eval/loss █▁▁
wandb:            eval/runtime ▁▇█
wandb: eval/samples_per_second █▂▁
wandb:   eval/steps_per_second █▂▁
wandb:             train/epoch ▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇█████
wandb:       train/global_step ▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇███
wandb:         train/grad_norm ▁▁▂▂█▂▃▃▄▃▄▂▃▄▂▂▃▂▁▂▂▃▃▂▁▁▁▃▃▁▁▁▁▁▁▁▁▁▁▁
wandb:     train/learning_rate █████▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 1
wandb:                 eval/f1 1
wandb:               eval/loss 0.00011
wandb:            eval/runtime 4.7587
wandb: eval/samples_per_se

electra-base-mcq-v2 -> Local MAP@3 (held-out): 1.0000


wandb: updating run metadata
wandb: uploading history steps 0-0, summary, console lines 0-0
wandb: 
wandb: Run history:
wandb: MAP@3 ▁
wandb: 
wandb: Run summary:
wandb: MAP@3 1
wandb: 
wandb: 🚀 View run electra-base-mcq-v2 at: https://wandb.ai/varnitchourasiya27-indian-institute-of-technology-madras/23f3000843-t22026/runs/s5tt6kim
wandb: ⭐️ View project at: https://wandb.ai/varnitchourasiya27-indian-institute-of-technology-madras/23f3000843-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260723_095313-s5tt6kim/logs


In [11]:
def build_submission(test_df, predict_fn, out_path="submission.csv"):
    preds = test_df.apply(predict_fn, axis=1)
    sub   = pd.DataFrame({"id": test_df["id"], "prediction": preds})
    sub.to_csv(out_path, index=False)
    return sub

submission = build_submission(test, electra_predict_fn)
submission.head()

,id,prediction
0,1,A D C
1,2,B E A
2,3,B D A
3,4,E C D
4,5,C B A
